In [1]:
# 08_optimization.ipynb
# Combinatorial-optimization generalisations solved to optimality: multidimensional
# 0/1 knapsack, submodular saturation (exact piecewise-linear MILP) vs greedy,
# robust knapsack (Bertsimas-Sim Gamma), and the exact epsilon-constraint Pareto.
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupKFold
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,"axes.titlesize":1})
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
rng=np.random.default_rng(0)
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)
def wpct(v,wt,p):
    m=~np.isnan(v); vs=np.sort(v[m]); ww=wt[m][np.argsort(v[m])]; return np.interp(p,np.cumsum(ww)/ww.sum(),vs)

from scipy.optimize import milp, LinearConstraint, Bounds, linprog
stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat")); sch,_=pyreadstat.read_sas7bdat(str(DATA/"bcgkori3.sas7bdat"))
USED=["NISB","S_HOMLIT","S_INTNET","S_HISEI","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT",
      "S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD","S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE","S_EXCOMP"]
for c in USED: stu.loc[(stu[c]>=995)|(stu[c]<=-995),c]=np.nan
stu.loc[(stu.NISB<-6)|(stu.NISB>6),"NISB"]=np.nan; stu["CIL5"]=stu[PVS].mean(1)
LEV=["S_LRNINTO","S_ICTPOSS","S_GENCLASS","S_LRNSAFE"]; CTX=["NISB","S_HISEI","S_HOMLIT","S_INTNET","S_EXCOMP","S_GENEFF","S_SPECEFF","S_ICTPOSG","S_ICTFUT","S_ACMULT","S_LRNINTS","S_CTCLS","S_SPECLASS","S_ICTSTUD"]
FE=LEV+CTX; d=stu[stu.TOTWGTS.notna()&stu.IDSCHOOL.notna()].copy(); W=d.TOTWGTS.to_numpy()
Xb=d[FE].to_numpy(float); Xc=Xb.copy()
for j,L in enumerate(LEV):
    t=wpct(Xb[:,j],W,0.75); col=Xc[:,j]; Xc[:,j]=np.where(np.isnan(col),col,np.maximum(col,t))
m=HistGradientBoostingRegressor(max_iter=400,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=0).fit(Xb,d.PV1CIL.to_numpy(float),sample_weight=W)
d["dCIL"]=m.predict(Xc)-m.predict(Xb)
g=d.groupby("IDSCHOOL").apply(lambda z:pd.Series(dict(mN=z.TOTWGTS.sum(),delta=np.average(z.dCIL,weights=z.TOTWGTS),
    nisb=np.average(z.NISB.fillna(z.NISB.mean()),weights=z.TOTWGTS),n=len(z),
    sd=np.sqrt(np.cov(z.dCIL,aweights=z.TOTWGTS)) if len(z)>1 else 0.0))).reset_index()
g["se"]=g.sd/np.sqrt(g.n); N=len(g); val=(g.mN*g.delta).to_numpy(); c1=g.mN.to_numpy()
shh=sch[["IDSCHOOL","C_HINRES"]].copy(); shh.loc[(shh.C_HINRES>=995)|(shh.C_HINRES<=-995),"C_HINRES"]=np.nan
g=g.merge(shh,on="IDSCHOOL",how="left"); hz=((g.C_HINRES-np.nanmean(g.C_HINRES))/np.nanstd(g.C_HINRES)).fillna(0)
c2=(g.mN*(1.0+0.5*np.clip(hz,-1.5,1.5))).to_numpy(); B1=0.33*c1.sum(); B2=0.33*c2.sum()
g["bucket"]=pd.qcut(g.nisb,3,labels=[0,1,2]).astype(int); bk=g.bucket.to_numpy()
def mkp(vals,integer):
    r=milp(c=-vals,constraints=LinearConstraint(np.vstack([c1,c2]),-np.inf,[B1,B2]),integrality=(np.ones(N) if integer else np.zeros(N)),bounds=Bounds(0,1),options={"disp":False}); return -r.fun,r.x
lp,lpx=mkp(val,False); ip,ipx=mkp(val,True)
pd.DataFrame([dict(units=N,resources=2,LP_relax=round(lp),MILP_0_1=round(ip),integrality_gap_pct=round(100*(lp-ip)/lp,3))]).to_csv(TAB/"opt_axis1_mkp.csv",index=False)
print(f"AXIS 1 MKP: LP={lp:,.0f} MILP={ip:,.0f} gap={100*(lp-ip)/lp:.3f}%")
# AXIS 2 exact submodular via piecewise-linear MILP vs greedy
beta=0.6; phi=lambda z: np.power(np.maximum(z,1e-9),beta); dphi=lambda z: beta*np.power(np.maximum(z,1e-9),beta-1)
nb=3; nv=N+nb; rows=[np.concatenate([c1,np.zeros(nb)]),np.concatenate([c2,np.zeros(nb)])]; rhs=[B1,B2]
for b in range(nb):
    Z=val[bk==b].sum()
    for zk in np.linspace(Z*0.02,Z,10):
        row=np.zeros(nv); row[np.where(bk==b)[0]]=-dphi(zk)*val[bk==b]; row[N+b]=1.0; rows.append(row); rhs.append(phi(zk)-dphi(zk)*zk)
res=milp(c=np.concatenate([np.zeros(N),-np.ones(nb)]),constraints=LinearConstraint(np.array(rows),-np.inf,rhs),
    integrality=np.concatenate([np.ones(N),np.zeros(nb)]),bounds=Bounds(np.zeros(nv),np.concatenate([np.ones(N),[phi(val[bk==b].sum()) for b in range(nb)]])),options={"disp":False})
xopt=res.x[:N].round().astype(bool); fval=lambda S: sum(phi(val[(bk==b)&S].sum()) for b in range(nb))
def greedy():
    S=np.zeros(N,bool)
    while True:
        best=None
        for i in np.where(~S)[0]:
            T=S.copy(); T[i]=True
            if c1[T].sum()>B1 or c2[T].sum()>B2: continue
            gg=(fval(T)-fval(S))/((c1[i]/B1+c2[i]/B2)/2+1e-9)
            if best is None or gg>best[1]: best=(i,gg)
        if best is None: break
        S[best[0]]=True
    return S
Sg=greedy(); f_opt=fval(xopt); f_g=fval(Sg); f_mod=fval(ipx.round().astype(bool))
pd.DataFrame([dict(submodular_OPT=round(f_opt,1),greedy=round(f_g,1),greedy_ratio_to_OPT=round(f_g/f_opt,3),
    guarantee=round(1-1/np.e,3),OPT_over_value_greedy_pct=round(100*(f_opt-f_mod)/f_mod,2))]).to_csv(TAB/"opt_axis2_submodular.csv",index=False)
print(f"AXIS 2 submodular: greedy/OPT={f_g/f_opt:.3f} (>= {1-1/np.e:.3f}); OPT over value-greedy={100*(f_opt-f_mod)/f_mod:.2f}%")
# AXIS 3 robust Gamma
z=1.0; dev=z*(g.mN*g.se).to_numpy(); nom=ip; nomx=ipx.round(); rr=[]
def robust(G):
    best=None
    for th in np.unique(np.append(dev,0.0)):
        adj=val-np.maximum(dev-th,0.0); _,xx=mkp(adj,True); xx=xx.round(); obj=val@xx-np.maximum(dev-th,0.0)@xx-G*th
        if best is None or obj>best[0]: best=(obj,xx)
    return best
for G in [0,5,15,30,60]:
    _,rx=robust(G); realized=val@rx
    rr.append(dict(Gamma=G,price_of_robustness_pct=round(100*(nom-realized)/nom,2),units=int(rx.sum()),overlap=int(((nomx>0.5)&(rx>0.5)).sum())))
A3=pd.DataFrame(rr); A3.to_csv(TAB/"opt_axis3_robust.csv",index=False); print("AXIS 3 robust:\n",A3.to_string(index=False))
fig,ax=plt.subplots(figsize=(4.2,3.1)); ax.plot(A3.Gamma,A3.price_of_robustness_pct,"o-",color="0.2")
ax.set_xlabel("Uncertainty budget Gamma"); ax.set_ylabel("Price of robustness (%)"); savefig(fig,"fig_price_robustness")
# AXIS 4 exact epsilon-constraint Pareto (subgroups)
R=pd.read_csv(TAB/"rev_m2_ci.csv"); Dg=R.dCIL.to_numpy(); bg=R.base.to_numpy(); mg=R.wN.to_numpy(); grp=list(R.grp)
hi=np.array([x.startswith("High") for x in grp]); lo=np.array([x.startswith("Low") for x in grp]); MH=mg[hi].sum(); ML=mg[lo].sum()
gred=(lo*mg*Dg/ML)-(hi*mg*Dg/MH); eff=mg*Dg; cap=mg; budget=0.33*mg.sum(); nG=len(grp)
emax=-linprog(c=-gred,A_ub=np.array([cap]),b_ub=[budget],bounds=[(0,1)]*nG,method="highs").fun
par=lambda e:(lambda r:(-r.fun if r.success else np.nan))(linprog(c=-eff,A_ub=np.vstack([cap,-gred]),b_ub=[budget,-e],bounds=[(0,1)]*nG,method="highs"))
P=pd.DataFrame([(e,par(e)) for e in np.linspace(0,emax,21)],columns=["gap_reduction_min","efficiency"]).dropna()
P.to_csv(TAB/"opt_axis4_pareto.csv",index=False)
print(f"AXIS 4 Pareto: max feasible gap reduction={emax:.2f}; price of fairness over full range={100*(P.efficiency.iloc[0]-P.efficiency.iloc[-1])/P.efficiency.iloc[0]:.2f}%")


AXIS 1 MKP: LP=3,162,700 MILP=3,158,549 gap=0.131%


AXIS 2 submodular: greedy/OPT=0.975 (>= 0.632); OPT over value-greedy=0.11%
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


... (38 outputs left out)

HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();
HighsMipSolverData::transformNewIntegerFeasibleSolution tmpSolver.run();


AXIS 3 robust:
  Gamma  price_of_robustness_pct  units  overlap
     0                     0.00     51       51
     5                     0.16     53       48
    15                     0.16     53       48
    30                     0.25     54       47
    60                     0.28     54       47


AXIS 4 Pareto: max feasible gap reduction=18.40; price of fairness over full range=0.00%
